# Lesson 16 Lab — Serving LoRA Adapters

**Puzzle:** Can one base model safely serve many task adapters without duplicating all weights?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

LoRA keeps a shared base checkpoint and applies small low-rank deltas per request. The memory advantage is attractive, but adapter identity, rank limits, tokenizer compatibility, scheduling, and dynamic-load security become service concerns.


## 0. Predict before running

1. Estimate one adapter's bytes at rank 16.
2. Probe `--enable-lora` and rank-related arguments.
3. Explain why this lab cannot claim adapter output correctness.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab probes LoRA API and CLI support, builds a transparent low-rank memory ledger for the local architecture, and validates request-routing identities. It does not fabricate a trained adapter.

- LoRA shares base weights but adds per-adapter state.
- Adapter name and immutable revision belong in the request contract.
- Dynamic loading expands the filesystem and authorization boundary.


## 2. Derive the mechanism

For a matrix `W`, LoRA adds `ΔW = B A` with rank `r`; storage scales with `r(in+out)` rather than `in×out`. vLLM can batch requests associated with different adapters while sharing the base weights, subject to configured rank and resident-adapter limits. The adapter path and name become executable inputs.

### Mechanism at a glance

```mermaid
flowchart LR
  B["shared base weights W"] --> Y["linear output"]
  R["request adapter ID"] --> A["load A and B factors"]
  X["activation x"] --> Y
  A --> D["x B A low-rank delta"]
  D --> Y
  Y --> O["adapter-specific result"]
```

### Walk it step by step

1. **Freeze the base.** All tenants reference one immutable base revision.
2. **Resolve an authorized adapter.** Map the request name to a signed local artifact.
3. **Apply the low-rank delta.** Schedule adapter-specific factors beside shared weights.
4. **Test isolation.** Verify quality, residency limits, eviction, and authorization.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 16
LESSON_TITLE = 'Serving LoRA Adapters'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260828
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | full base-model duplication per task |
| Candidate | one base model plus rank-16 adapter deltas |
| Held constant | model geometry, dtype, target modules, rank, adapter count, and installed vLLM |
| Measurements | estimated bytes, compression ratio, API symbols, CLI flags, and native-adapter execution status |
| Evidence | `compatibility-probe` |

**Experiment:** Calculate adapter storage and inspect the installed LoRA request/config surface with explicit missing-native evidence.


## 5. Inspect the experiment code

The code derives matrix dimensions from config and counts only declared target projections. It records theoretical bytes separately from package/API availability.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); hidden=int(cfg["hidden_size"]); intermediate=int(cfg["intermediate_size"])
layers=int(cfg["num_hidden_layers"]); rank=16; shapes=[(hidden,hidden)]*4+[(intermediate,hidden)]*2+[(hidden,intermediate)]
adapter_params=layers*sum(rank*(din+dout) for dout,din in shapes); adapter_bytes=adapter_params*2
weight_bytes=sum(p.stat().st_size for p in MODEL.glob("*.safetensors")); _,serve_help=cli_help("serve")
api=False
try:
    from vllm.lora.request import LoRARequest
    api=inspect.isclass(LoRARequest)
except Exception: pass
metrics={"rank":rank,"target_matrices_per_layer":len(shapes),"estimated_adapter_parameters":adapter_params,
         "estimated_adapter_bytes":adapter_bytes,"base_weight_bytes":weight_bytes,
         "adapter_to_base_ratio":adapter_bytes/weight_bytes,"api":{"lora_request":api},
         "cli":{"enable_lora":"--enable-lora" in serve_help,"max_lora_rank":"--max-lora-rank" in serve_help,
                "max_loras":"--max-loras" in serve_help},"native_adapter_executed":False}
analysis=(f"The rank-{rank} seven-projection estimate is {adapter_bytes:,} BF16 bytes "
          f"({metrics['adapter_to_base_ratio']:.2%} of weights); request API/enable flag="
          f"{api}/{metrics['cli']['enable_lora']}. No trained adapter behavior was fabricated.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Estimated adapter bytes | 39,223,296 bytes |
| Base weight bytes | 3,087,467,144 bytes |
| Storage ratio | 1.27% |
| LoRA request API | yes |
| Enable flag | no |
| Native adapter executed | no |


## 7. Explain the result

The rank-16 seven-projection estimate is 39,223,296 BF16 bytes (1.27% of weights); request API/enable flag=True/False. No trained adapter behavior was fabricated.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 16, "title": 'Serving LoRA Adapters', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The low-rank ledger explains why adapters are small; native behavioral and performance claims remain pending without a real adapter artifact.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 16,
  "title": "Serving LoRA Adapters",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260828
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "rank": 16,
    "target_matrices_per_layer": 7,
    "estimated_adapter_parameters": 19611648,
    "estimated_adapter_bytes": 39223296,
    "base_weight_bytes": 3087467144,
    "adapter_to_base_ratio": 0.012704036729985685,
    "api": {
      "lora_request": true
    },
    "cli": {
      "enable_lora": false,
      "max_lora_rank": false,
      "max_loras": false
    },
    "native_adapter_executed": false
  },
  "analysis": "The rank-16 seven-projection estimate is 39,223,296 BF16 bytes (1.27% of weights); request API/enable flag=True/False. No trained adapter behavior was fabricated.",
  "conclusion": "The low-r

## 9. Make the bounded decision

> The low-rank ledger explains why adapters are small; native behavioral and performance claims remain pending without a real adapter artifact.

**Acceptance/rollback:** Enable multi-adapter serving only after signed adapter artifacts, task quality, isolation, load/unload, concurrency, and rollback tests pass.

**Failure analysis:** Real PEFT checkpoints contain configuration and may target a different module set. Runtime memory includes buffers, and an unauthorized local path can expose arbitrary artifacts.


## 10. Extend the evidence

Create or obtain a versioned adapter, hash it, run baseline/adapter requests through `LoRARequest`, and stress simultaneous adapter residency and eviction.

The full boundary and references are in [`README.md`](README.md).
